# 02 – M0 (W2-05), M1 (W2-06), M1-h (W2-07)

Hướng dẫn từng bước: `docs/huong_dan/W2-05.md`, `W2-06.md`, `W2-07.md`.
Ô có chữ **TODO** là phần bạn tự viết. Chỉ chọn siêu tham số trên **dev**, chỉ đánh giá test ở bước cuối.

In [ ]:
# Ô 0 – chuẩn bị môi trường Colab (chạy mỗi lần mở notebook)
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/NLI'      # thư mục Drive chung có data/processed
!git clone -q https://github.com/quanuiter/NLI.git /content/NLI 2>/dev/null || git -C /content/NLI pull -q
%cd /content/NLI
!pip install -q -r requirements.txt
import os
for sub in ['data/processed', 'tools', 'results']:
    target = os.path.join(DRIVE_DIR, sub); os.makedirs(target, exist_ok=True)
    if os.path.islink(sub): continue
    if os.path.isdir(sub):
        !cp -rn {sub}/. {target}/ && rm -rf {sub}
    os.symlink(target, sub)

In [ ]:
from collections import Counter
import re
import numpy as np
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import f1_score
from src.data import load_scenario
from src.embeddings import tokenize
from src.evaluate import evaluate_and_save, summarize
from src.utils import set_seed
set_seed(42)
data = {s: load_scenario(s) for s in ['S1', 'S2']}
{s: {k: (len(v) if not isinstance(v, dict) else {t: len(x) for t, x in v.items()}) for k, v in d.items()} for s, d in data.items()}

## M0 – nhãn đa số (W2-05): chạy được ngay

In [ ]:
for s, d in data.items():
    majority = Counter(d['train']['label']).most_common(1)[0][0]
    print(s, 'nhãn đa số của train:', majority)
    for test_name, df in d['test'].items():
        evaluate_and_save(df['label'], [majority] * len(df), model='M0', scenario=s, test_set=test_name,
                          seed=42, config={'majority_label': majority}, ids=df['id'])

## M1 – TF-IDF + Logistic Regression / Naive Bayes (W2-06)

Mỗi câu được biến thành vector TF-IDF **riêng** cho premise và hypothesis rồi ghép lại, cộng thêm các
**đặc trưng so sánh hai câu** do bạn tự viết. Chỉ TF-IDF thì mô hình không biết hai câu liên quan nhau thế nào
(trên ViANLI macro-F1 dev chỉ khoảng 0,2 – thấp hơn đoán bừa).

In [ ]:
NEG = re.compile(r'\b(không|chưa|chẳng|chả|đâu có)\b', re.IGNORECASE)

def compare_features(df):
    """Trả về ma trận (số mẫu × số đặc trưng), mỗi đặc trưng >= 0 (để dùng được cả Naive Bayes)."""
    rows = []
    for p, h in zip(df['premise'], df['hypothesis']):
        tp, th = set(t.lower() for t in tokenize(p)), set(t.lower() for t in tokenize(h))
        overlap = len(tp & th) / max(1, len(th))          # tỉ lệ từ của H có trong P
        # TODO a: số từ phủ định trong P và trong H: len(NEG.findall(p)), len(NEG.findall(h))
        # TODO b: chênh lệch độ dài |len(p) - len(h)| (theo âm tiết), chia cho 50 để về cùng thang đo
        # TODO c: H có số nào không xuất hiện trong P? (re.findall(r'\d+', ...)) -> 1.0 / 0.0
        rows.append([overlap])                             # TODO: thêm các đặc trưng a, b, c vào danh sách
    return np.array(rows)

def make_vectorizers():
    return (TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
            TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True))

def features(df, vp, vh, fit=False):
    P = vp.fit_transform(df['premise']) if fit else vp.transform(df['premise'])
    H = vh.fit_transform(df['hypothesis']) if fit else vh.transform(df['hypothesis'])
    return sp.hstack([P, H, sp.csr_matrix(compare_features(df))]).tocsr()

In [ ]:
# Chọn C của Logistic Regression trên DEV cho S2 (làm tương tự cho S1)
s = 'S2'
d = data[s]
vp, vh = make_vectorizers()
X_train = features(d['train'], vp, vh, fit=True)
X_dev = features(d['dev'], vp, vh)
for C in [0.1, 1, 10]:
    model = LogisticRegression(C=C, max_iter=2000).fit(X_train, d['train']['label'])
    print(f'C={C}  dev macro-F1 = {f1_score(d["dev"]["label"], model.predict(X_dev), average="macro"):.3f}')

**TODO d:** làm tương tự với `MultinomialNB(alpha=...)`, thử alpha ∈ {0.1, 0.5, 1.0}. Ghi bảng kết quả dev (LR và NB) vào báo cáo.

**TODO e:** chọn mô hình + siêu tham số tốt nhất trên dev cho **mỗi** kịch bản, huấn luyện lại rồi đánh giá test:
```python
for test_name, df in d['test'].items():
    evaluate_and_save(df['label'], best_model.predict(features(df, vp, vh)), model='M1', scenario=s,
                      test_set=test_name, seed=42, config={'clf': 'LR', 'C': best_C, 'features': '...'},
                      ids=df['id'], premise=df['premise'], hypothesis=df['hypothesis'])
```

In [ ]:
# TODO d, e

## M1-h – chỉ dùng hypothesis (W2-07)

**TODO f:** giống M1 nhưng **chỉ** TF-IDF của hypothesis (không premise, không `compare_features`), chọn C trên dev,
lưu với `model='M1h'`. So sánh với M0 và M1: nếu M1-h cao hơn hẳn M0 thì dữ liệu "lộ nhãn" qua hypothesis.

In [ ]:
# TODO f

In [ ]:
summarize().query("model in ['M0', 'M1', 'M1h']")